# 02 — Silver: NL + Marketplace (M6 — re-scoped)

**Owner:** M6 • **Tier:** Light • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)

## Scope — re-scoped per new allocation
- **Source tables (8 in this notebook; 5 more in 02_silver_sellers stay with M2):**
  - `nl_*` (5): nl_listings, nl_listing_events, nl_user_accounts, nl_categories, nl_event_types
  - `ts_*` mkt transactional (3): ts_marketplace_orders, ts_seller_listings, ts_fulfilment_events
- **T7 (NL listing confidence) is M2's** (defended weights = heavy work — see 02_silver_sellers.ipynb cells M2 will add)
- **Silver transforms owned (in this notebook):** apply T1/T2/T3
- **Anomalies in this notebook:** A4, A13 (your other 2 — A15, A16 — are in cs_/rv_ data, see hint scripts below)

## Hint script
- **A4 (NL seller-marked-sold counted as confirmed revenue):** in `nl_listing_events`, count events with `event_type='SELLER_SOLD'`. These are NOT confirmed transactions. Flag `NL_SELLER_SOLD_AS_REVENUE` with `metric_certainty='ESTIMATED'` (these events feed B6's GMV model with weight 0.60 per the instructor's catalog — they're signal, not noise).
- **B6 row labelling (paired with M2's Section 7 formula write-up):** every row in `silver_nl_listing_events` whose `event_type IN ('SELLER_SOLD','PHN_REVEAL','CHAT','OFFER_ACC')` carries `ESTIMATED_NL_GMV` in `anomaly_reason_code` (comma-separated alongside any other applicable code).
- **A13 (image hash reused across sellers):** group `nl_listings` by `image_hash`, count distinct sellers per hash. Hashes used by ≥2 sellers indicate coordination. Flag affected listings with `IMAGE_HASH_REUSED`.

## Hands-on-of-everything checklist (covers your overall, not just this notebook)
- [ ] Bronze read
- [ ] Silver write: 8 tables here + 5 more (rv/cs) — total 13
- [ ] Gold dims: dim_seller (SCD1) — pulls in M2's seller_trust_score; dim_return_reason (static)
- [ ] Gold facts: fact_classified_listing_event, fact_customer_review (the latter in your cs/rv work)
- [ ] Anomalies: A4, A13 (here) + A15, A16 (in your rv_/cs_ Silver — add cells to 02_silver_sellers.ipynb is wrong; they belong with the source data; create a small ad-hoc notebook OR add cells here for cs_*/rv_*)
- [ ] Bus Matrix: 2 fact rows + dim_seller cells
- [ ] Report: Sections 1, 2, 7, 8, 9

> **Note:** for `rv_*` and `cs_*` Silver work + A15/A16 detection, you can either add cells to this notebook or create a separate scratch notebook. Either is fine — what matters is the Silver tables land and the anomalies are flagged.

## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Cell 3 — `silver_nl_user_accounts` + `silver_nl_listings`

**TODO:**
- Read both Bronze tables; parse datetimes (iso_timestamp with T)
- SK on `user_id` and `listing_id`
- For listings: **flag A13** by computing `image_hash → distinct seller count`. Listings whose `image_hash` is shared across ≥ 2 sellers get `IMAGE_HASH_REUSED`, status `FLAGGED_ANOMALY`, certainty `INFERRED`.
- For listings: **flag A12** with the strict detection — compute window over (seller_id, image_hash) ordered by created_datetime; if a previous listing for the same key had `status IN ('SOLD','EXPIRED')` within prior 30 days, flag current listing `RELISTED_AFTER_SOLD`.
- Write both Silver tables

In [ ]:
# TODO M6: NL listings with A12 + A13 detection

## Cell 4 — `silver_nl_listing_events` + A4 detection

**TODO:**
- Read `nl_listing_events`, parse iso_timestamp
- SK on `(listing_id, event_seq)`
- **Flag A4:** rows where `event_type='SELLER_SOLD'` get `NL_SELLER_SOLD_AS_REVENUE`, status `FLAGGED_ANOMALY`, certainty **`ESTIMATED`** (not UNRELIABLE — per the instructor's catalog these events feed B6's GMV model with weight 0.60; they're segregated, not discarded).
- **B6 row labelling:** every row whose `event_type IN ('SELLER_SOLD','PHN_REVEAL','CHAT','OFFER_ACC')` additionally carries `ESTIMATED_NL_GMV` in `anomaly_reason_code` (comma-separated). Status stays `CLEAN`, certainty stays `ESTIMATED` — these are platform-recorded signals being labelled as B6 inputs, not anomalies in themselves.
- All other NL events: `metric_certainty='CONFIRMED'` for purely-platform engagement (views, contacts) where the event is the truth.
- Write

In [ ]:
# TODO M6: NL events + A4

## Cell 5 — T7: `silver_listing_confidence`

Aggregate engagement signals per listing → composite confidence score.

**TODO:**
- For each listing, compute:
  - `listing_age_days` from created_datetime to today
  - `response_rate` = chats_responded / chats_initiated (from events)
  - `price_reasonableness` = 1 - |asking_price - category_median| / category_median, clamped to [0,1]
  - `relist_count_norm` = your A12 detection count / max
  - `report_count_norm` = ts_safety_reports about this listing / max
- Apply weights from formula in header markdown (or your justified alternative)
- Output `silver_listing_confidence` with: listing_id (FK), confidence_score, plus the 5 input components for transparency
- Flag any listing with `confidence_score < 0.30` as `LISTING_LOW_CONFIDENCE`

In [ ]:
# TODO M6: T7 implementation

## Cell 6 — Marketplace Silver: listings, orders, fulfilment

**TODO:**
- Read `ts_seller_listings`, `ts_marketplace_orders`, `ts_fulfilment_events`
- Parse dates per format hints
- SK on natural keys
- Apply T2 status normalisation
- Join to `silver_product_master` (M3) for canonical_product_key on listings
- Write three Silver tables

In [ ]:
# TODO M6: marketplace silvers

## Cell 7 — Acceptance check

Before requesting PR review:
1. `silver_nl_listings` has rows flagged `IMAGE_HASH_REUSED` (multi-ring detection)
2. `silver_nl_listings` has rows flagged `RELISTED_AFTER_SOLD` (small count)
3. `silver_nl_listing_events` has rows flagged `NL_SELLER_SOLD_AS_REVENUE` (proportional to SOLD listings)
4. `silver_listing_confidence` has scores in [0.0, 1.0] and at least some `LISTING_LOW_CONFIDENCE` flags
5. All 4 mandatory anomaly columns populated everywhere